# Step 09 — Merge LLM Batch Results

**Input:** `data/output/llm_predictions/predictions_with_geometry.gpkg` (and optionally numbered `-N.gpkg` variants from multiple runs)  
**Output:** `data/output/09_llm_predictions_merged.gpkg`

**Bug fix:** The original notebook hardcoded exactly 11 file names (df, df1…df10).
This version uses a dynamic glob so it works regardless of how many runs were done.

**Merge strategy:** Files are processed in order. For any `gml_id` that appears in multiple files,
the LATER file wins (replacing the earlier prediction). This is correct because reruns improve quality.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import re
import ast
import pandas as pd
import geopandas as gpd
print('Config loaded')

## 1. Discover all prediction GPKG files (dynamic glob)

In [ ]:
# Find predictions_with_geometry.gpkg and any numbered variants
pred_files = sorted(
    LLM_PREDICTIONS_DIR.glob('predictions_with_geometry*.gpkg'),
    key=lambda p: (
        int(re.search(r'-(\d+)\.gpkg$', p.name).group(1))
        if re.search(r'-(\d+)\.gpkg$', p.name) else -1
    )
)

# Also check for legacy naming pattern from original pipeline
legacy_files = sorted(
    LLM_PREDICTIONS_DIR.glob('final_predictions_with_geometry_rerun*.gpkg'),
    key=lambda p: (
        int(re.search(r'-(\d+)\.gpkg$', p.name).group(1))
        if re.search(r'-(\d+)\.gpkg$', p.name) else -1
    )
)

all_files = pred_files + [f for f in legacy_files if f not in pred_files]
print(f'Found {len(all_files)} prediction file(s):')
for f in all_files:
    print(f'  {f.name}')

## 2. Merge: later files replace earlier predictions for same gml_id

In [ ]:
merged_gdf = None
stats      = []

for path in all_files:
    new_gdf = gpd.read_file(path)
    if len(new_gdf) == 0:
        print(f'Skipped empty: {path.name}')
        continue

    new_gdf['gml_id'] = new_gdf['gml_id'].astype(str)
    new_gdf = new_gdf.drop_duplicates(subset='gml_id', keep='last')

    if merged_gdf is None:
        merged_gdf = new_gdf.copy()
        stats.append({'file': path.name, 'added': len(merged_gdf), 'replaced': 0})
        continue

    existing_ids = set(merged_gdf['gml_id'])
    new_ids      = set(new_gdf['gml_id'])
    replaced     = existing_ids & new_ids
    added        = new_ids - existing_ids

    merged_gdf = merged_gdf[~merged_gdf['gml_id'].isin(replaced)]
    merged_gdf = pd.concat([merged_gdf, new_gdf], ignore_index=True)
    merged_gdf = gpd.GeoDataFrame(merged_gdf, geometry='geometry', crs=new_gdf.crs)
    stats.append({'file': path.name, 'added': len(added), 'replaced': len(replaced)})

pd.DataFrame(stats)

## 3. Clean mid_labels column

In [ ]:
def clean_mid_labels(val):
    if pd.isna(val) or val is None: return []
    if isinstance(val, list):
        return [str(x).strip() for x in val if str(x).strip() in TARGET_MID_LABELS]
    try:
        parsed = ast.literal_eval(str(val))
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip() in TARGET_MID_LABELS]
    except Exception:
        pass
    return [x.strip() for x in str(val).strip('[]').split(',') if x.strip() in TARGET_MID_LABELS]

merged_gdf['mid_label'] = merged_gdf['mid_labels'].apply(clean_mid_labels)
print(f'Total merged: {len(merged_gdf):,}')
print(f'Rows with mid_label: {merged_gdf["mid_label"].apply(bool).sum():,}')
print(f'Rows with bosserhof_class: {merged_gdf["bosserhof_class"].notna().sum():,}')

## 4. Save

In [ ]:
merged_gdf.to_file(LLM_MERGED_FILE, driver='GPKG')
print(f'Saved {len(merged_gdf):,} rows → {LLM_MERGED_FILE}')